In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# =========================
# 1. データ読み込み
# =========================
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

# 目的変数を 0/1 に変換
train['Heart Disease'] = train['Heart Disease'].map({'Presence': 1, 'Absence': 0})

# 説明変数
features = train.columns.drop(['id', 'Heart Disease'])
X = train[features]
y = train['Heart Disease']
X_test = test[features]

# 数値・カテゴリの分割（CatBoost 用）
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

# 念のため：カテゴリとして扱いたい整数カラムを指定してもOK
# 例: cat_cols += ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
# 今回はすべて数値として入っている前提で進める

# =========================
# 2. KFold ロジスティック回帰（B）
# =========================
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_lr = np.zeros(len(train))
test_pred_lr = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_trn_scaled = scaler.fit_transform(X_trn)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    lr = LogisticRegression(max_iter=500)
    lr.fit(X_trn_scaled, y_trn)
    
    oof_lr[val_idx] = lr.predict_proba(X_val_scaled)[:, 1]
    test_pred_lr += lr.predict_proba(X_test_scaled)[:, 1] / kf.n_splits

print("Logistic Regression CV AUC:", roc_auc_score(y, oof_lr))

# =========================
# 3. CatBoost モデル（A）
# =========================
from catboost import CatBoostClassifier, Pool

# CatBoost は Pool 形式を使うと便利
train_pool = Pool(X, label=y, cat_features=cat_cols if len(cat_cols) > 0 else None)
test_pool = Pool(X_test, cat_features=cat_cols if len(cat_cols) > 0 else None)

# ベースライン CatBoost
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 0,
    'depth': 6,
    'learning_rate': 0.05,
    'iterations': 1000
}

cb_model = CatBoostClassifier(**cb_params)
cb_model.fit(train_pool)

oof_cb = cb_model.predict_proba(train_pool)[:, 1]
test_pred_cb = cb_model.predict_proba(test_pool)[:, 1]

print("CatBoost (baseline) AUC (train):", roc_auc_score(y, oof_cb))

# =========================
# 4. Optuna で CatBoost をチューニング（D）
# =========================
import optuna

def objective(trial):
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': 0,
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'iterations': 1000
    }
    
    oof = np.zeros(len(train))
    
    for trn_idx, val_idx in kf.split(X, y):
        X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        
        train_pool_cv = Pool(X_trn, label=y_trn, cat_features=cat_cols if len(cat_cols) > 0 else None)
        val_pool_cv = Pool(X_val, label=y_val, cat_features=cat_cols if len(cat_cols) > 0 else None)
        
        model = CatBoostClassifier(**params)
        model.fit(train_pool_cv, eval_set=val_pool_cv, use_best_model=True)
        
        oof[val_idx] = model.predict_proba(val_pool_cv)[:, 1]
    
    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)  # 時間に応じて増減

print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params.update({
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 0,
    'iterations': 1000
})

cb_best = CatBoostClassifier(**best_params)
cb_best.fit(train_pool)

oof_cb_best = cb_best.predict_proba(train_pool)[:, 1]
test_pred_cb_best = cb_best.predict_proba(test_pool)[:, 1]

print("CatBoost (tuned) AUC (train):", roc_auc_score(y, oof_cb_best))

# =========================
# 5. アンサンブル（E）
# =========================
# ここでは：
# - KFold ロジスティック回帰
# - チューニング後 CatBoost
# の平均をとる

test_pred_ensemble = (test_pred_lr + test_pred_cb_best) / 2

# =========================
# 6. 提出ファイル作成
# =========================
submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': test_pred_ensemble
})

submission.to_csv('submission.csv', index=False)
print("submission.csv を作成しました。")